In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Modul de testare locală Qiskit Runtime

<span id="test-locally"></span>


<details>
<summary><b>Versiuni de pachete</b></summary>

Codul de pe această pagină a fost dezvoltat folosind următoarele cerințe.
Îți recomandăm să folosești aceste versiuni sau unele mai noi.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-aer~=0.17
```
</details>
Folosește modul de testare locală (disponibil începând cu `qiskit-ibm-runtime` v0.22.0 sau mai recent) pentru a testa programele înainte de a le ajusta fin și de a le trimite pe hardware cuantic real. După ce folosești modul de testare locală pentru a verifica programul, tot ce trebuie să schimbi este numele Backend-ului pentru a-l rula pe un QPU.

Pentru a folosi modul de testare locală, specifică unul dintre Backend-urile false din ``qiskit_ibm_runtime.fake_provider`` sau specifică un Backend Qiskit Aer când instanțiezi un primitiv Qiskit Runtime sau o Session.

- **Backend-uri false**: [Backend-urile false](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider) din ``qiskit_ibm_runtime.fake_provider`` imită comportamentul QPU-urilor IBM&reg; folosind instantanee ale QPU-urilor. Instantaneele QPU conțin informații importante despre QPU, cum ar fi harta de cuplare, porțile de bază și proprietățile qubiților, care sunt utile pentru testarea Transpiler-ului și pentru efectuarea de simulări cu zgomot ale QPU-ului. Modelul de zgomot din instantaneu este aplicat automat în timpul simulării.
- **Simulatorul Aer**: Simulatoarele din [Qiskit Aer](/guides/simulate-with-qiskit-aer) oferă simulări de performanță mai ridicată, care pot gestiona Circuit-uri mai mari și [modele de zgomot personalizate](/guides/build-noise-models). O listă de opțiuni pentru metodele de simulare este disponibilă atunci când folosești `AerSimulator` în modul de testare locală. Vezi [exemplul modului de simulare Clifford](#clifford-sim), care demonstrează cum să simulezi eficient Circuit-uri Clifford cu un număr mare de qubiți.

    <details>
    <summary>**Listă de metode de simulare disponibile din Qiskit Aer**</summary>

    Consultă documentația [`AerSimulator`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator) pentru mai multe informații.

    * ``"automatic"``: Metoda de simulare implicită. Selectează automat metoda de simulare, în funcție de Circuit și de modelul de zgomot.

    * ``"statevector"``: O simulare densă a vectorului de stare care poate eșantiona rezultatele măsurătorilor din Circuit-uri *ideale* cu toate măsurătorile la sfârșitul Circuit-ului. Pentru simulări cu zgomot, fiecare shot eșantionează aleatoriu un Circuit cu zgomot din modelul de zgomot.

    * ``"density_matrix"``: O simulare a matricei de densitate care poate eșantiona rezultatele măsurătorilor din Circuit-uri *cu zgomot* cu toate măsurătorile la sfârșitul Circuit-ului.

    * ``"stabilizer"``: Un simulator eficient al stării stabilizatoare Clifford care poate simula Circuit-uri Clifford cu zgomot dacă toate erorile din modelul de zgomot sunt și ele erori Clifford.

    * ``"extended_stabilizer"``: Un simulator aproximativ pentru Circuit-uri Clifford + T, bazat pe descompunerea stării într-o stare de rang-stabilizator. Numărul de termeni crește odată cu numărul de porți non-Clifford (T).

    * ``"matrix_product_state"``: Un simulator de vector de stare bazat pe rețele tensoriale care folosește o reprezentare de tip Matrix Product State (MPS) pentru stare. Aceasta poate fi realizată cu sau fără truncherea dimensiunilor legăturilor MPS, în funcție de opțiunile simulatorului. Comportamentul implicit este fără trunchiere.

    * ``"unitary"``: O simulare densă a matricei unitare a unui Circuit ideal. Aceasta simulează matricea unitară a Circuit-ului în sine, nu evoluția unei stări cuantice inițiale. Această metodă poate simula doar Gate-uri; nu suportă măsurători, resetări sau zgomot.

    * ``"superop"``: O simulare densă a matricei de superoperator a unui Circuit ideal sau cu zgomot. Aceasta simulează matricea de superoperator a Circuit-ului în sine, nu evoluția unei stări cuantice inițiale. Această metodă poate simula Gate-uri și resetări ideale și cu zgomot, dar nu suportă măsurători.

    * ``"tensor_network"``: O simulare bazată pe rețele tensoriale care suportă atât vectorul de stare, cât și matricea de densitate. În prezent, aceasta este disponibilă doar pentru GPU și este accelerată prin utilizarea API-urilor cuQuantum `cuTensorNet`.
    </details>

> **Note:** - Poți specifica toate opțiunile Qiskit Runtime în modul de testare locală. Totuși, toate opțiunile cu excepția shots sunt ignorate când rulează pe un simulator local.
> - Se recomandă să instalezi Qiskit Aer înainte de a folosi Backend-urile false sau simulatoarele Aer, rulând `pip install qiskit-aer`. Backend-urile false folosesc simulatoarele Aer pe fundal, dacă sunt disponibile, pentru a beneficia de performanța acestora.


## Exemplu cu Backend-uri false

In [1]:
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime.fake_provider import FakeManilaV2

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using FakeManilaV2
fake_manila = FakeManilaV2()
pm = generate_preset_pass_manager(backend=fake_manila, optimization_level=1)
isa_qc = pm.run(qc)

# You can use a fixed seed to get fixed results.
options = {"simulator": {"seed_simulator": 42}}
sampler = Sampler(mode=fake_manila, options=options)

result = sampler.run([isa_qc]).result()

## Exemple cu AerSimulator
Exemplu cu Session-uri, fără zgomot:

In [2]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import Session, SamplerV2 as Sampler

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using AerSimulator.
# Session syntax is supported but ignored because local mode doesn't support sessions.
aer_sim = AerSimulator()
pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
with Session(backend=aer_sim) as session:
    sampler = Sampler(mode=session)
    result = sampler.run([isa_qc]).result()

Pentru a simula cu zgomot, specifică un QPU (hardware cuantic) și trimite-l către Aer. Aer construiește un model de zgomot bazat pe datele de calibrare de la acel QPU și instanțiază un Backend Aer cu acel model. Dacă preferi, poți [construi un model de zgomot](/guides/build-noise-models).

> **Caution:** Un QPU poate fi afectat de diferite tipuri de zgomot. Modelul de zgomot Qiskit Aer folosit aici simulează doar o parte dintre ele și, prin urmare, este probabil să fie mai puțin sever decât zgomotul de pe un QPU real.
> 
> Pentru detalii despre ce erori sunt incluse la inițializarea unui model de zgomot dintr-un QPU, consultă referința API Aer [`NoiseModel`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.noise.NoiseModel.html#qiskit_aer.noise.NoiseModel.from_backend).

Exemplu cu zgomot:

In [3]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler, QiskitRuntimeService

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

service = QiskitRuntimeService()

# Specify a QPU to use for the noise model
real_backend = service.backend("ibm_fez")
aer = AerSimulator.from_backend(real_backend)

# Run the sampler job locally using AerSimulator.
pm = generate_preset_pass_manager(backend=aer, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer)
result = sampler.run([isa_qc]).result()

<span id="clifford-sim"></span>
### Simularea Clifford
Deoarece Circuit-urile Clifford pot fi simulate eficient cu rezultate verificabile, simularea Clifford este un instrument foarte util. Pentru un exemplu detaliat, vezi [Simularea eficientă a Circuit-urilor stabilizatoare cu primitivele Qiskit Aer](/guides/simulate-stabilizer-circuits).

Exemplu:

In [4]:
import numpy as np
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import SamplerV2 as Sampler

n_qubits = 500  # <---- note this uses 500 qubits!
circuit = efficient_su2(n_qubits)
circuit.measure_all()

rng = np.random.default_rng(1234)
params = rng.choice(
    [0, np.pi / 2, np.pi, 3 * np.pi / 2],
    size=circuit.num_parameters,
)

# Tell Aer to use the stabilizer (Clifford) simulation method
aer_sim = AerSimulator(method="stabilizer")

pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer_sim)
result = sampler.run([isa_qc]).result()

## Pași următori
> **Tip:** - Consultă [exemple detaliate de primitive.](/guides/primitives-examples)
>     - Citește [Migrarea la primitivele V2](/guides/v2-primitives).
>     - Exersează cu primitivele parcurgând [lecția despre funcția cost](/learning/courses/variational-algorithm-design/cost-functions) din IBM Quantum Learning.
>     - Află cum să transpui local în secțiunea [Transpiler](/guides/transpile).
>     - Încearcă tutorialul [Compararea setărilor Transpiler-ului](/guides/circuit-transpilation-settings#compare-transpiler-settings).